# Preprocessing Pipeline: Climate-Socioeconomic Panel

**Objective**: Transform cleaned EDA data into modeling-ready format with engineered features, proper scaling, and time-aware train/test split.

**Inputs**: 
- `../data/processed/df_cleaned.csv` (output from EDA notebook)

**Outputs**:
- `X_train`, `X_test`, `y_train`, `y_test` (saved to `../data/processed/`)
- `scaler.pkl`, `feature_names.json` (saved to `../models/`)
- Engineered feature documentation

**Key Steps**:
1. Load cleaned data
2. Implement feature engineering (per-capita metrics, lags, ratios, interactions)
3. Handle scaling with RobustScaler
4. TimeSeriesSplit: train on 1900-2009, test on 2010-2023
5. Save artifacts for reproducibility

**Note**: All transformations must be fit on training data only to avoid leakage.

In [1]:
# IMPORTS & CONFIGURATION 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json
import warnings
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
import os

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Paths
DATA_CLEAN = '../data/processed/df_cleaned.csv'
DATA_PROCESSED = '../data/processed/'
MODELS_DIR = '../models/'
FIGURES_DIR = '../outputs/figures/'

# Ensure output directories exist
for dir_path in [DATA_PROCESSED, MODELS_DIR, FIGURES_DIR]:
    os.makedirs(dir_path, exist_ok=True)

print("Imports and paths configured successfully.")

Imports and paths configured successfully.


In [ ]:
# LOAD DATA & INITIAL SETUP 
# Load cleaned dataset
df = pd.read_csv(DATA_CLEAN)
# Basic verification
print("Loaded dataset info:")
print(f"Shape: {df.shape}")
print(f"Year range: {df['Year'].min()} - {df['Year'].max()}")
print(f"Countries: {df['Country'].nunique()}")

TARGETS = ['Temperature_Anomaly', 'CO2_Emissions']

# Sort by Country and Year (CRITICAL for correct lag/rolling calculations)
df = df.sort_values(['Country', 'Year']).reset_index(drop=True)

# Temporal split to prevent future data leakage
SPLIT_YEAR = 2010
df_train = df[df['Year'] < SPLIT_YEAR].copy()
df_test = df[df['Year'] >= SPLIT_YEAR].copy()

print(f"\nTemporal split (Year < {SPLIT_YEAR}):")
print(f"Train set: {len(df_train)} rows | Years: {df_train['Year'].min()}-{df_train['Year'].max()}")
print(f"Test set:  {len(df_test)} rows | Years: {df_test['Year'].min()}-{df_test['Year'].max()}")

Loaded dataset info:
Shape: (23797, 26)
Year range: 1900 - 2023
Countries: 195

Temporal split (Year < 2010):
Train set: 21106 rows | Years: 1900-2009
Test set:  2691 rows | Years: 2010-2023


In [9]:
#FEATURE ENGINEERING
import sys
sys.path.append('..')
from src.feature_engineering import engineer_full_pipeline
print(f"Total columns after engineering: {df.shape[1]}")
print(f"Columns: {df.columns.tolist()}")

Total columns after engineering: 47
Columns: ['Country', 'Year', 'Air_Pollution_Index', 'Arctic_Ice_Extent', 'Average_Rainfall', 'Average_Temperature', 'Biodiversity_Index', 'CO2_Emissions', 'Deforestation_Rate', 'Energy_Consumption_Per_Capita', 'Extreme_Weather_Events', 'Forest_Area', 'Fossil_Fuel_Usage', 'GDP', 'Industrial_Activity', 'Methane_Emissions', 'Ocean_Acidification', 'Per_Capita_Emissions', 'Policy_Score', 'Population', 'Renewable_Energy_Usage', 'Sea_Level_Rise', 'Solar_Energy_Potential', 'Temperature_Anomaly', 'Urbanization', 'Waste_Management', 'GDP_per_Capita', 'CO2_per_Capita', 'Renewable_Ratio', 'Policy_GDP_Interaction', 'Decade', 'GDP_lag1', 'GDP_lag5', 'GDP_roll5_mean', 'GDP_roll5_std', 'CO2_Emissions_lag1', 'CO2_Emissions_lag5', 'CO2_Emissions_roll5_mean', 'CO2_Emissions_roll5_std', 'Renewable_Energy_Usage_lag1', 'Renewable_Energy_Usage_lag5', 'Renewable_Energy_Usage_roll5_mean', 'Renewable_Energy_Usage_roll5_std', 'Fossil_Fuel_Usage_lag1', 'Fossil_Fuel_Usage_lag5',

In [10]:
#  PREPARE FEATURES, SCALING, AND SAVE ARTIFACTS 

# 1. Define target and feature columns
TARGETS = ['Temperature_Anomaly', 'CO2_Emissions']
DROP_COLS = ['Country', 'Year'] + TARGETS  # Exclude identifiers and prediction targets

# Re-split engineered data by year (ensures temporal alignment after feature creation)
df_train = df[df['Year'] < SPLIT_YEAR].copy()
df_test = df[df['Year'] >= SPLIT_YEAR].copy()

# Separate features (X) and targets (y)
X_train = df_train.drop(columns=DROP_COLS)
y_train = df_train[TARGETS]
X_test = df_test.drop(columns=DROP_COLS)
y_test = df_test[TARGETS]

# 2. Handle any remaining missing values (robust fallback)
# Impute test data with TRAIN statistics to prevent data leakage
train_medians = X_train.median()
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

# 3. Apply RobustScaler (fit ONLY on training data)
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# 4. Verify and print summary
print("Data preparation complete:")
print(f"Train features: {X_train_scaled.shape}")
print(f"Test features:  {X_test_scaled.shape}")
print(f"Number of features used for modeling: {X_train_scaled.shape[1]}")
print(f"Targets: {TARGETS}")

# 5. Save artifacts for reproducibility and next notebook
X_train_scaled.to_csv(f'{DATA_PROCESSED}X_train.csv', index=False)
X_test_scaled.to_csv(f'{DATA_PROCESSED}X_test.csv', index=False)
y_train.to_csv(f'{DATA_PROCESSED}y_train.csv', index=False)
y_test.to_csv(f'{DATA_PROCESSED}y_test.csv', index=False)
joblib.dump(scaler, f'{MODELS_DIR}scaler.pkl')

print(f"\nArtifacts saved successfully to {DATA_PROCESSED} and {MODELS_DIR}")

Data preparation complete:
Train features: (18181, 43)
Test features:  (2691, 43)
Number of features used for modeling: 43
Targets: ['Temperature_Anomaly', 'CO2_Emissions']

Artifacts saved successfully to ../data/processed/ and ../models/


In [11]:
#  SANITY CHECKS & FINAL REPORT 

# 1. Check for missing values (Critical for ML)
assert X_train_scaled.isnull().sum().sum() == 0, "Error: Missing values detected in X_train"
assert X_test_scaled.isnull().sum().sum() == 0, "Error: Missing values detected in X_test"

# 2. Verify shapes
print(f"Training Features Shape: {X_train_scaled.shape}")
print(f"Testing Features Shape:  {X_test_scaled.shape}")
print(f"Training Targets Shape:  {y_train.shape}")
print(f"Testing Targets Shape:   {y_test.shape}")

# 3. Inspect Feature List
print(f"\nTotal Modeling Features: {X_train_scaled.shape[1]}")
print("Features used for modeling:")
# Print in a cleaner list format
for col in X_train_scaled.columns:
    print(f"  - {col}")

# 4. Verify Target Split (Temporal consistency check)
print(f"\nTarget Summary (Train vs Test):")
print(f"  Train Temp Anomaly Mean: {y_train['Temperature_Anomaly'].mean():.4f}")
print(f"  Test  Temp Anomaly Mean: {y_test['Temperature_Anomaly'].mean():.4f}")
print(f"  Train CO2 Mean: {y_train['CO2_Emissions'].mean():.2e}")
print(f"  Test  CO2 Mean: {y_test['CO2_Emissions'].mean():.2e}")

print("\n=== PREPROCESSING COMPLETE ===")
print("Data is scaled, split, and saved.")
print("Proceed to 'notebooks/03_model_training_tuning.ipynb'")

Training Features Shape: (18181, 43)
Testing Features Shape:  (2691, 43)
Training Targets Shape:  (18181, 2)
Testing Targets Shape:   (2691, 2)

Total Modeling Features: 43
Features used for modeling:
  - Air_Pollution_Index
  - Arctic_Ice_Extent
  - Average_Rainfall
  - Average_Temperature
  - Biodiversity_Index
  - Deforestation_Rate
  - Energy_Consumption_Per_Capita
  - Extreme_Weather_Events
  - Forest_Area
  - Fossil_Fuel_Usage
  - GDP
  - Industrial_Activity
  - Methane_Emissions
  - Ocean_Acidification
  - Per_Capita_Emissions
  - Policy_Score
  - Population
  - Renewable_Energy_Usage
  - Sea_Level_Rise
  - Solar_Energy_Potential
  - Urbanization
  - Waste_Management
  - GDP_per_Capita
  - CO2_per_Capita
  - Renewable_Ratio
  - Policy_GDP_Interaction
  - Decade
  - GDP_lag1
  - GDP_lag5
  - GDP_roll5_mean
  - GDP_roll5_std
  - CO2_Emissions_lag1
  - CO2_Emissions_lag5
  - CO2_Emissions_roll5_mean
  - CO2_Emissions_roll5_std
  - Renewable_Energy_Usage_lag1
  - Renewable_Energy_Us